# Customer Churn Analysis (Python, SQLite)

## Project description
Analysis of a subscription business: which customers cancel, and what do they have in common? Data is read from a SQLite database with 3 tables (customer, subscription, support), cleaned and merged with Pandas, then analysed with KPIs, charts and pivot tables.

## Business problem
The company wants to reduce churn (customers cancelling). Questions answered here:
1. What are the churn rate and retention rate?
2. Which plans, subscription types and states churn the most?
3. Do complaints and escalations go together with churn?
4. How much monthly revenue is lost, and how much is at risk?

## Dataset note (important)
`customer_churn.db` has 21 original sample customers (`is_synthetic = 0`) plus 150 generated customers (`is_synthetic = 1`) added for practice, so the analysis has more rows to work with. The generated rows copy the patterns of the original ones, so the results show how the analysis works. They are not real business findings.

Built while following a tutorial by Rishabh Mishra, then extended with my own fixes and written conclusions.

In [ ]:
# importing Libraries
import numpy as np
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sns
import sqlite3 as sql

In [ ]:
# import database
conn = sql.connect('customer_churn.db')

sql_query = """
            select name
            from sqlite_master
            where type = 'table'

"""

tables = pd.read_sql(sql_query,conn)

# create dataframe for each tables

for tables_name in tables['name']:
    df = pd.read_sql(f"select * from {tables_name}",conn)
    globals()[f"df_{tables_name}"] = df
    print(f"Created Dataframe:df_{tables_name}")

conn.close()

In [ ]:
# print tables names and column names
conn = sql.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\n Table names{table_name}")
    # get column information
    column_query = f"PRAGMA table_info({(table_name)});"
    column = pd.read_sql(column_query,conn)
    print("columns:")
    print(column['name'].tolist())

#close connection

conn.close()

# Data Cleaning

In [ ]:
df_db_customer.head()

In [ ]:
df_db_customer.tail()

In [ ]:
df_db_customer.info()

In [ ]:
df_db_customer.isnull().sum()

In [ ]:
# a.drop columns :- pincode , interest
# b data standardization on column gender
# c.change dob datatype object to datetime
# d. change name to customer_name
# e . fix missing values in country column

In [ ]:
df_db_customer['country'].unique()

In [ ]:
# a.drop columns :- pincode , interest
# df_db_customer.drop(df_db_customer.columns[-2:],axis = 1) 

# we can also do it like this  df_db_customer.columns[6:]

df_db_customer = df_db_customer.drop(["interests","pincode"],axis = 1)

In [ ]:
df_db_customer

In [ ]:
df_db_customer['gender'].unique()

In [ ]:
# b data standardization on column gender

df_db_customer = df_db_customer.replace({"Men":"Male","Women":"Female"})

In [ ]:
df_db_customer.head()

In [ ]:
df_db_customer.tail()

In [ ]:
df_db_customer['gender'].unique() 

In [ ]:
# c.change dob datatype object to datetime
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

In [ ]:
df_db_customer.info()

In [ ]:
df_db_customer['dob'].dtype

In [ ]:
# d. change name to customer_name

df_db_customer = df_db_customer.rename(columns={"name" : "customer_name"})

In [ ]:
df_db_customer.columns

In [ ]:
# e . fix missing values in country column

df_db_customer['country'].unique()

In [ ]:
df_db_customer['country'].isna().sum()

In [ ]:
df_db_customer[df_db_customer['country'].isna()][['state','country']]

In [ ]:
manual_map = {'Delhi' : 'India', 'Telangana' : 'India', 'Kathmandu' : 'Nepal'}

df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(manual_map))

In [ ]:
df_db_customer['country'].unique()

In [ ]:
df_db_customer['country'].isna().sum()

In [ ]:
df_db_subscription.head()

In [ ]:
df_db_subscription.tail()

In [ ]:
df_db_subscription.info()

In [ ]:
date_column = ['subscription_start_date','renewal_date','cancellation_date']

df_db_subscription[date_column] = df_db_subscription[date_column].apply(pd.to_datetime)

In [ ]:
# fix spelling mistake in the data: Refferal -> Referral
df_db_subscription['subscription_type'] = df_db_subscription['subscription_type'].replace({'Refferal':'Referral'})

In [ ]:
df_db_subscription.info()

In [ ]:
df_db_support.head()

In [ ]:
df_db_support = df_db_support.drop(columns=['col_1','comment'])

In [ ]:
df_db_support.head()

In [ ]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

In [ ]:
df_db_support.info()

# 3. Feature Engineering and Data Analysis

In [ ]:
df_db_subscription.head(3)

In [ ]:
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)

In [ ]:
df_db_subscription.head()

In [ ]:
df = (df_db_subscription.merge(df_db_customer, on = 'customerid', how = 'left')
                   .merge(df_db_support, on = 'customerid', how = 'left') )

In [ ]:
df.shape

In [ ]:
df_db_subscription.shape

In [ ]:
df.head()

In [ ]:
df_db_subscription['customerid'].nunique()

In [ ]:
df_db_customer['customerid'].nunique()

In [ ]:
df_db_support['customerid'].nunique()

In [ ]:
df_db_support['customerid'].size

In [ ]:
df_db_support

In [ ]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [ ]:
df_db_support.head()

In [ ]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep='last')

In [ ]:
df_db_support['customerid'].size

In [ ]:
# first fix duplicates table then merge df
df = (df_db_subscription.merge(df_db_customer, on = 'customerid', how = 'left')
                   .merge(df_db_support, on = 'customerid', how = 'left') )

In [ ]:
df.head(2)

In [ ]:
df.shape

In [ ]:
df.to_csv('Exported_churn_data.csv', index = False)

In [ ]:
df.columns

In [ ]:
# Churn rate 
churn_rate = round(df['churn_flag'].mean()*100,2)
print("Churn_rate:",churn_rate,'%')

In [ ]:
# Retention rate 
retention_rate = 100 - churn_rate 

print("retention_rate:",retention_rate,'%')

In [ ]:
# Churn by plan type

churn_plan_type = round(df.groupby('plan_type')['churn_flag'].mean()*100,2).reset_index()
churn_plan_type

In [ ]:
df.columns

In [ ]:
# 4.a churn by state + sum(revenue) + count of users

churn_by_state = df.groupby('state').agg(
                                         churn_users = ('churn_flag','mean'),
                                         total_revenue = ('monthly_charges','sum'),
                                         total_customer = ('customerid','count')
)
                                
churn_by_state

In [ ]:
#churn by subscription type + sum(revenue) + count of users

churn_by_subs = df.groupby('subscription_type').agg(
                                                    churn_by_subd = ('churn_flag',"mean"),
                                                    total_revenue = ('monthly_charges','sum'),
                                                    total_count = ('customerid','count')
)

churn_by_subs

In [ ]:
# 5 Avg revenue per user

avg_revenue = df['monthly_charges'].mean()
print("Avg_Revenue is: ",round(avg_revenue,2))

In [ ]:
df.head(2)

In [ ]:
# calculate customer age 

In [ ]:
# 6 AVG customer Tenure
# count of days users has used our services : cancellation date else current date

today = pd.Timestamp('2025-12-31')   # fixed snapshot date, so the result does not change every day

df['Tenure_days'] = np.where(

    df['cancellation_date'].notna(),

 (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['Tenure_days'].mean()

print("Avg Tenure(days) = ", avg_tenure)

In [ ]:
# 7 revenue lost - monthly charges of customers who already churned
revenue_lost = df.loc[df['churn_flag']==1,'monthly_charges'].sum()
print("Monthly revenue lost from churned customers: ", round(revenue_lost,2))

In [ ]:
# Escalation rate

Escalation_rate = (df['escalations']=='Y').mean()*100

print("Escalation Rate: ",round(Escalation_rate,2),"%")

In [ ]:
# 9. Avg Complaint per users

avg_complains = df['complaint_count'].sum() / df['customerid'].nunique()
print("Avg complaint per user: ",round(avg_complains,2))

In [ ]:
# 10. correlation Esclation vs churn

df['escalations'] = np.where(df['escalations']=='Y',1,0) # encoding str to int type 
corr_df = df[['escalations','churn_flag']].dropna()

# df[]
correlation = corr_df['escalations'].corr(df['churn_flag'])
print("correlation between escalation vs churn is : ",round(correlation,2))


In [ ]:
# 11. Churn Risk - create column using existing col

condition = [(df['churn_score']<50),
            (df['churn_score']>=50) & (df['churn_score']<70),
            (df['churn_score']>=70)
            ]
choices = ['low','med','high']

df['churn_risk'] = np.select(condition,choices, default = 'unknown')

In [ ]:
# 7b revenue at risk - customers still active but with medium or high churn risk
at_risk = df[(df['churn_flag']==0) & (df['churn_risk'].isin(['med','high']))]
print("Active customers at risk:", len(at_risk))
print("Monthly revenue at risk:", round(at_risk['monthly_charges'].sum(),2))

In [ ]:
df.columns

In [ ]:
df.head()

# Data Visualization

In [ ]:
df_visual = df.copy()

In [ ]:
df_visual.shape

In [ ]:
df_visual.columns

In [ ]:
df_visual['cancellation_month'] = df_visual['cancellation_date'].dt.to_period('M')

# churn trend

churn_trend = df_visual[df_visual['churn_flag']==1].groupby('cancellation_month').size()

plt.figure(figsize=(8,3))
plt.plot(churn_trend.index.astype(str),churn_trend.values,color='red', marker='o', linestyle='dashed', linewidth=2, markersize=12)

plt.title("Monthly Churn Trend")
plt.xlabel("Months")
plt.ylabel("Churned Customers")
plt.show()

In [ ]:
# 2. churn by plan type

churn_plan = df_visual.groupby('plan_type')['churn_flag'].mean()

# colors = ['yellow','purple','blue']
colors = plt.cm.Set2(np.linspace(0, 1, len(churn_plan)))
plt.figure(figsize=(8,3))
plt.bar(churn_plan.index, churn_plan.values, color = colors)

plt.title("Churned by Plan")
plt.xlabel("Plan")
plt.ylabel("Churn rate")
plt.show()

In [ ]:
# 3. churn by state 

churn_state = df_visual.groupby('state')['churn_flag'].mean()

colors = plt.cm.Set2(np.linspace(0,1 , len(churn_state)))

plt.figure(figsize =(8,3))
plt.barh(churn_state.index, churn_state.values , color = colors)

plt.title("Churn By State")
plt.xlabel("Churn rate")
plt.ylabel("State")
plt.show()

In [ ]:
# Using Seaborn

In [ ]:
df_visual.columns

In [ ]:
# df_encoded
df_visual[['plan_type', 'contract_type','churn_score','churn_flag', 'churn_risk', 'escalations']].head()

In [ ]:
df_encoded = df_visual[['plan_type', 'contract_type','churn_score','churn_flag', 'churn_risk', 'escalations']].copy()

# ordered codes (not alphabetical), so the correlations make sense
df_encoded['plan_type'] = df_encoded['plan_type'].map({'Basic':0, 'Standard':1, 'Premium':2})
df_encoded['contract_type'] = df_encoded['contract_type'].map({'Monthly':0, 'Annual':1})
df_encoded['churn_risk'] = df_encoded['churn_risk'].map({'low':0, 'med':1, 'high':2})

In [ ]:
df_encoded.head()

In [ ]:
sns.heatmap(df_encoded.corr(),annot= True)

In [ ]:
#  catplot/ facegrid plot - multi - dim comparision

sns.catplot(data = df_visual,
            x = 'plan_type',
            y = 'monthly_charges',
            hue = 'gender',
            col = 'churn_risk'
            )

In [ ]:
# Pivot Table

In [ ]:
pd.pivot_table(
        df_visual,
        index = 'plan_type',
        values = 'churn_flag',
        aggfunc= 'mean'
    
)

In [ ]:
df_visual.columns

In [ ]:
pd.pivot_table(
         data = df_visual,
         index = 'plan_type',
         values = ['monthly_charges','customerid','churn_flag'],
         aggfunc= {
             'monthly_charges': 'sum',
             'customerid' : 'nunique',
             'churn_flag' : 'mean'
         }
    
)

# Key Findings and Recommendations

*Based on the expanded dataset (171 customers, of which 150 are generated). Read these as a demonstration of the method, not as real business results.*

## Findings
- **Churn rate is 23.39%** and retention is 76.61%.
- **Basic plan churns the most:** 52% (40 customers), against 19% for Standard and 9% for Premium.
- **Referral customers churn more:** 42%, against 19% for Paid and 15% for Organic.
- **Escalated complaints go together with churn:** correlation 0.59. Customers with no complaint are counted as "no escalation".
- **Revenue:** monthly revenue lost from churned customers is 515.60. 30 active customers have medium or high churn risk, and their monthly revenue is 524.70.
- Average customer tenure is about 1,421 days (up to 31 Dec 2025).

## Recommendations
1. Focus retention offers on Basic plan customers, for example a discount or an upgrade trial.
2. Review how referral customers are onboarded, because many leave.
3. Contact customers quickly after an escalated complaint.
4. Call the 30 active customers with medium or high churn risk before their renewal date.

## Limitations
- 150 of the 171 rows are generated, so results mirror the patterns of the 21 original rows.
- Correlation shows association, not cause.
- One monthly charge of 92.99 in the original data looks like a typing mistake (other Premium charges are about 20 to 23). It was not changed.